## For SLEA controls within 80K NGN2 derived neurons
- 'C_SLEA'

In [1]:
from importlib import reload
import pandas as pd
import sys
sys.path.append('../helpful_functions')
import helpful_functions as hf
reload(hf)

<module 'helpful_functions' from '/home/kisa/coding/80K_MPRA/80K-Analysis/07_quality_control/notebooks/control_metadata/../helpful_functions/helpful_functions.py'>

In [2]:
# helpful functions

# column names
col_name = 'name'
col_header = 'name'
col_sequence = 'sequence'
col_category = 'category'
col_class = 'class'
col_source = 'source'
col_ref = 'ref'
col_chr = 'chr'
col_start = 'start'
col_end = 'end'
col_strand = 'strand'
col_variant_class = 'variant_class'
col_variant_pos = 'variant_pos'
col_SPDI = 'SPDI'
col_allele = 'allele'
col_info = 'info'
my_col_ref_base = 'tmp_ref_base'
my_col_alt_base = 'tmp_alt_base'


def get_start_end_strand_control(row):
    """
    Special cases to set chr, start, end and strand for control sequences from their header (because not in region bed)
    """
    name = row[col_name]
    row[col_ref] = 'GRCh37'
    row[col_class] = 'element inactive control'
    region_info = name.split(':chr')[1].split('|')[0]
    row[col_chr] = f"chr{region_info.split(':')[0]}"
    row[col_start] = int(region_info.split(':')[1].split('-')[0])
    row[col_end] = int(region_info.split(':')[1].split('-')[1])
    row[col_strand] = '.'
    return row


In [ ]:
import yaml

# config
config_path = "/home/kisa/coding/80K_MPRA/80K-Analysis/07_quality_control/notebooks/control_metadata/config_file.yaml"
with open(config_path) as conf:
    config = yaml.load(conf, Loader=yaml.FullLoader)
    conf.close()

input_fasta = config['design_file']
pre_metadata_df = hf.fasta_to_dataframe(input_fasta, columns=[col_name, col_sequence])
pre_metadata_df['tmp_label'] = pre_metadata_df[col_name].apply(lambda x: hf.get_label(x))
pre_metadata_df

synthetic_control_groups = ['C_SLEA']
# # filter for underscore parsable headers
pre_metadata_df_filtered = pre_metadata_df.loc[pre_metadata_df['tmp_label'].isin(synthetic_control_groups)].copy()
expected_number = pre_metadata_df_filtered.shape[0]

# add the columns of the metadata file
pre_metadata_df_filtered[col_category] = 'synthetic'
pre_metadata_df_filtered[col_class] = 'element inactive control'
pre_metadata_df_filtered[col_source] = 'NA'
pre_metadata_df_filtered[col_ref] = 'GRCh37'
pre_metadata_df_filtered[col_chr] = 'NA'
pre_metadata_df_filtered[col_start] = 'NA'
pre_metadata_df_filtered[col_end] = 'NA'
pre_metadata_df_filtered[col_strand] = 'NA'
pre_metadata_df_filtered[col_variant_class] = 'NA'
pre_metadata_df_filtered[col_variant_pos] = 'NA'
pre_metadata_df_filtered[col_SPDI] = 'NA'
pre_metadata_df_filtered[col_allele] = 'NA'
pre_metadata_df_filtered[col_info] = 'synthetically modified regions from hg18'

# parse regions from header
pre_metadata_df_filtered = pre_metadata_df_filtered.apply(get_start_end_strand_control, axis=1)
pre_metadata_df_filtered

# remove adapter from sequence (15bp of start and end):
pre_metadata_df_filtered[col_sequence] = pre_metadata_df_filtered[col_sequence].apply(lambda x: x[15:-15])

In [7]:
interesting_columns = [col_name, col_sequence, col_category, col_class, col_source, col_ref, col_chr, col_start, col_end, col_strand, col_variant_class, col_variant_pos, col_SPDI, col_allele, col_info]

# Write DataFrame to TSV file
pre_metadata_df_filtered[interesting_columns].to_csv('/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/mpra_metadata_format/SLEA/SLEA.metadata.tmp.tsv.gz', sep='\t', index=False, na_rep='NA', compression='gzip')
print('Number of expected sequences in the metadata file: ', expected_number)
print('Number of sequences in the metadata file: ', pre_metadata_df_filtered[interesting_columns].shape[0])
import os
os.system('zcat /data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/mpra_metadata_format/SLEA/SLEA.metadata.tmp.tsv.gz | sed "s/\'/\\"/g" | gzip -c > /data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/mpra_metadata_format/SLEA/SLEA.metadata.tsv.gz')

Number of expected sequences in the metadata file:  200
Number of sequences in the metadata file:  200


0